# Dynamic Pricing for Urban Parking Lots
### Summer Analytics 2025 — Consulting & Analytics Club × Pathway

The idea here is simple — parking lots charge a flat rate all day, which doesn't make much sense. A lot that's nearly full at noon should cost more than an empty one at 8am. So I built a pricing engine that adjusts prices based on real conditions.

I tried three different approaches, each a bit smarter than the last:

| Model | Description |
|-------|-------------|
| **Model 1** | Baseline Linear — price tracks occupancy rate |
| **Model 2** | Demand-Based — multi-feature demand function with normalization |
| **Model 3** | Competitive — adds geo-proximity and competitor awareness |

Data is streamed via **Pathway** and visualized in real time with **Bokeh**.

---
## Setup: Installing packages

In [ ]:
# Install required packages (run once)
!pip install pathway bokeh panel --quiet

---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import datetime
import math
import warnings
warnings.filterwarnings('ignore')

import pathway as pw

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Legend
from bokeh.layouts import gridplot, column
from bokeh.palettes import Category20, Spectral11
from bokeh.io import push_notebook
import panel as pn

output_notebook()
pn.extension()

print("✅ All imports successful.")

---
## Step 1 : Loading the data and Preprocessing

Load the CSV, combine date and time into one timestamp, and sort everything chronologically so the stream plays back in the right order.

In [ ]:
# ─────────────────────────────────────────────
# Load raw dataset
# ─────────────────────────────────────────────
df = pd.read_csv('dataset.csv')   # <-- Upload dataset.csv to your Colab session

# Combine date and time into a single Timestamp column
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# Sort chronologically and reset index
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Parking spaces: {df['SystemCodeNumber'].nunique()}")
print(f"Date range: {df['Timestamp'].min()} → {df['Timestamp'].max()}")
df.head()

Encoding the categorical columns — traffic and vehicle type need to be numbers for the demand formula.

- Trucks get weight 1.5 since they take up more space, bikes get 0.5
- High traffic actually reduces demand a bit — if roads are jammed, fewer people bother coming

In [ ]:
# ─────────────────────────────────────────────
# Encode categorical features as numeric
# ─────────────────────────────────────────────

# Vehicle type weight: truck demands most space → highest weight
vehicle_weight_map = {'bike': 0.5, 'cycle': 0.5, 'car': 1.0, 'truck': 1.5}
df['VehicleTypeWeight'] = df['VehicleType'].map(vehicle_weight_map).fillna(1.0)

# Traffic level: encode ordinal severity
traffic_map = {'low': 0, 'average': 1, 'high': 2}
df['TrafficLevel'] = df['TrafficConditionNearby'].map(traffic_map).fillna(1)

# Occupancy rate (0–1)
df['OccupancyRate'] = df['Occupancy'] / df['Capacity']

# Preview
print("Unique parking spaces:")
print(df['SystemCodeNumber'].unique())
df[['SystemCodeNumber','Timestamp','Capacity','Occupancy','OccupancyRate',
    'QueueLength','TrafficLevel','IsSpecialDay','VehicleTypeWeight']].head(8)

---
## Step 2 : Setting up for Pathway

Pathway reads from a CSV and replays it row by row to simulate live data. Saving the processed data here so Pathway can stream it.

In [ ]:
import os

os.makedirs('stream_data', exist_ok=True)

STREAM_COLS = [
    'Timestamp', 'SystemCodeNumber', 'Capacity', 'Occupancy',
    'OccupancyRate', 'QueueLength', 'TrafficLevel', 'IsSpecialDay',
    'VehicleTypeWeight', 'Latitude', 'Longitude'
]

for lot_id, group in df.groupby('SystemCodeNumber'):
    path = f'stream_data/{lot_id}.csv'
    group[STREAM_COLS].to_csv(path, index=False)

print(f"✅ Wrote {df['SystemCodeNumber'].nunique()} stream CSVs to stream_data/")

---
## Step 3 : Pricing logic

All three models start from **$10**. Each one adjusts up or down depending on demand.

I kept prices between $5 and $25 — no lot should randomly jump to $50 just because it's a busy afternoon.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  MODEL 1 — Baseline Linear Pricing
# ───────────────────────────────────────────────────────────────────────
#  Formula:  Price_{t+1} = Price_t + α * (Occupancy / Capacity)
#
#  Intuition: A simple linear increase proportional to how full the lot is.
#  Serves as a reference baseline for comparing advanced models.
# ═══════════════════════════════════════════════════════════════════════

BASE_PRICE = 10.0
ALPHA      = 5.0   # Maximum additional price at full occupancy

def model1_price(occupancy_rate: float, prev_price: float = BASE_PRICE) -> float:
    """
    Baseline linear model.
    - occupancy_rate: fraction of capacity filled (0.0 – 1.0)
    - prev_price    : price from the previous time step
    Returns the updated price, clamped to [BASE_PRICE * 0.5, BASE_PRICE * 2].
    """
    price = prev_price + ALPHA * occupancy_rate
    return float(np.clip(price, BASE_PRICE * 0.5, BASE_PRICE * 2))


# ═══════════════════════════════════════════════════════════════════════
#  MODEL 2 — Demand-Based Pricing
# ───────────────────────────────────────────────────────────────────────
#  Demand = α*(Occ/Cap) + β*QueueLen − γ*TrafficLvl + δ*IsSpecialDay
#           + ε*VehicleWeight
#
#  Price_t = BasePrice × (1 + λ × NormalizedDemand)
#
#  Coefficients are chosen so that:
#    • High occupancy and queue → demand ↑ → price ↑
#    • High congestion nearby   → demand ↓ (drivers may leave) → price ↓
#    • Special days             → demand ↑ → price ↑
#    • Heavier vehicles         → demand ↑ → price ↑
# ═══════════════════════════════════════════════════════════════════════

# Demand function coefficients
ALPHA_D = 0.6   # Weight on occupancy rate
BETA_D  = 0.1   # Weight on queue length
GAMMA_D = 0.05  # Weight on traffic level (negative impact)
DELTA_D = 0.3   # Weight on special day
EPS_D   = 0.1   # Weight on vehicle type

LAMBDA  = 0.5   # Price sensitivity to normalized demand

# Normalization bounds (empirically chosen from dataset)
DEMAND_MIN = -0.5
DEMAND_MAX =  1.5

def model2_demand(occ_rate: float, queue_len: float, traffic_lvl: float,
                  is_special: int, vehicle_weight: float) -> float:
    """Compute raw demand score from features."""
    demand = (
        ALPHA_D * occ_rate
        + BETA_D  * queue_len
        - GAMMA_D * traffic_lvl
        + DELTA_D * is_special
        + EPS_D   * vehicle_weight
    )
    return demand


def model2_price(occ_rate: float, queue_len: float, traffic_lvl: float,
                 is_special: int, vehicle_weight: float) -> float:
    """
    Demand-based model.
    Normalizes demand to [0, 1] then scales price in [BASE*0.5, BASE*2].
    """
    raw_demand = model2_demand(occ_rate, queue_len, traffic_lvl,
                               is_special, vehicle_weight)
    # Clip and normalize demand
    raw_clipped = np.clip(raw_demand, DEMAND_MIN, DEMAND_MAX)
    norm_demand = (raw_clipped - DEMAND_MIN) / (DEMAND_MAX - DEMAND_MIN)  # → [0,1]
    # Center around 0 so that average demand yields base price
    centered = norm_demand - 0.5  # → [-0.5, 0.5]
    price = BASE_PRICE * (1 + LAMBDA * centered * 2)  # price ∈ [BASE*0.5, BASE*1.5]
    return float(np.clip(price, BASE_PRICE * 0.5, BASE_PRICE * 2))


# ═══════════════════════════════════════════════════════════════════════
#  MODEL 3 — Competitive Pricing (with geo-proximity)
# ───────────────────────────────────────────────────────────────────────
#  Builds on Model 2 and additionally adjusts price based on what
#  nearby competitors are charging.
#
#  Competitive adjustment logic:
#    • lot_full & nearby_cheaper  → reduce price or suggest rerouting
#    • nearby_expensive           → can raise price while staying attractive
#    • nearby_similar             → no adjustment needed
# ═══════════════════════════════════════════════════════════════════════

PROXIMITY_RADIUS_KM = 2.0   # Only consider competitors within 2 km
REROUTE_THRESHOLD   = 0.95  # Suggest rerouting if lot is ≥ 95% full

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in kilometres between two lat/lon points."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi   = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def model3_price(lot_id: str, occ_rate: float, queue_len: float,
                 traffic_lvl: float, is_special: int, vehicle_weight: float,
                 lat: float, lon: float, competitor_prices: dict,
                 lot_locations: dict) -> tuple:
    """
    Competitive pricing model.

    Parameters
    ----------
    lot_id           : ID of the current parking lot
    occ_rate         : current occupancy / capacity
    queue_len        : number of waiting vehicles
    traffic_lvl      : 0=low, 1=avg, 2=high
    is_special       : 1 if special day else 0
    vehicle_weight   : weighted vehicle type
    lat, lon         : coordinates of this lot
    competitor_prices: {lot_id: current_price} for all other lots
    lot_locations    : {lot_id: (lat, lon)}

    Returns
    -------
    (final_price, reroute_suggestion)
    """
    # Base: Model 2 price
    base_price = model2_price(occ_rate, queue_len, traffic_lvl,
                               is_special, vehicle_weight)

    # Find nearby competitors
    nearby_prices = []
    nearest_lot   = None
    min_dist      = float('inf')

    for cid, cprice in competitor_prices.items():
        if cid == lot_id:
            continue
        clat, clon = lot_locations.get(cid, (lat, lon))
        dist = haversine_km(lat, lon, clat, clon)
        if dist <= PROXIMITY_RADIUS_KM:
            nearby_prices.append(cprice)
            if dist < min_dist:
                min_dist  = dist
                nearest_lot = cid

    reroute = None
    final_price = base_price

    if nearby_prices:
        avg_competitor = float(np.mean(nearby_prices))
        min_competitor = float(np.min(nearby_prices))

        if occ_rate >= REROUTE_THRESHOLD and min_competitor < base_price * 0.9:
            # Lot nearly full AND a cheaper option is close by → reroute
            reroute = nearest_lot
            # Reduce price slightly to retain borderline cases
            final_price = base_price * 0.95

        elif avg_competitor > base_price * 1.1:
            # Competitors are significantly more expensive → we can charge more
            adjustment = min((avg_competitor - base_price) * 0.3, base_price * 0.2)
            final_price = base_price + adjustment

        elif avg_competitor < base_price * 0.9:
            # Competitors are cheaper → reduce price to stay competitive
            adjustment = (base_price - avg_competitor) * 0.3
            final_price = base_price - adjustment

    final_price = float(np.clip(final_price, BASE_PRICE * 0.5, BASE_PRICE * 2.5))
    return final_price, reroute


print("✅ Pricing functions defined.")

# ─── Quick sanity checks ───
print(f"\nModel 1 (occ_rate=0.9): ${model1_price(0.9):.2f}")
print(f"Model 2 (occ=0.9, queue=5, traffic=high, special=1, truck):  "
      f"${model2_price(0.9, 5, 2, 1, 1.5):.2f}")
print(f"Model 2 (occ=0.2, queue=0, traffic=low,  special=0, bike):   "
      f"${model2_price(0.2, 0, 0, 0, 0.5):.2f}")

---
## Step 4 : Pathway stream setup

Defining the schema so Pathway knows what columns to expect, then loading the data as a stream. `replay_csv` replays the file row by row at a set rate — simulating sensors sending data in real time.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Define Pathway schema — mirrors the columns we saved in stream_data/
# ─────────────────────────────────────────────────────────────────────────

class ParkingSchema(pw.Schema):
    Timestamp        : str
    SystemCodeNumber : str
    Capacity         : int
    Occupancy        : int
    OccupancyRate    : float
    QueueLength      : int
    TrafficLevel     : int
    IsSpecialDay     : int
    VehicleTypeWeight: float
    Latitude         : float
    Longitude        : float

print("✅ ParkingSchema defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Load data stream via Pathway's replay_csv (simulates real-time ingest)
#  input_rate=500 → process ~500 rows/second (adjust as needed)
# ─────────────────────────────────────────────────────────────────────────

# NOTE: We use the combined CSV for the stream (all lots together)
df[STREAM_COLS].to_csv('parking_stream_all.csv', index=False)

stream = pw.demo.replay_csv(
    'parking_stream_all.csv',
    schema=ParkingSchema,
    input_rate=500
)

print("✅ Stream loaded.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Parse timestamps and extract day-level key for windowing
# ─────────────────────────────────────────────────────────────────────────

FMT = "%Y-%m-%d %H:%M:%S"

stream_with_time = stream.with_columns(
    t   = stream.Timestamp.dt.strptime(FMT),
    day = stream.Timestamp.dt.strptime(FMT).dt.strftime("%Y-%m-%dT00:00:00"),
)

print("✅ Timestamps parsed.")

### Model 1 window

Group by lot + day, get the average occupancy rate for that window, plug into the linear formula.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  ░░░  MODEL 1 PATHWAY WINDOW  ░░░
#
#  Aggregate over a 1-day tumbling window per lot.
#  Compute max/avg occupancy rate → linear price.
# ─────────────────────────────────────────────────────────────────────────

model1_window = (
    stream_with_time
    .windowby(
        pw.this.t,
        instance=pw.this.SystemCodeNumber + "_" + pw.this.day,  # partition per lot per day
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t             = pw.this._pw_window_end,
        lot_id        = pw.reducers.any(pw.this.SystemCodeNumber),
        occ_rate_max  = pw.reducers.max(pw.this.OccupancyRate),
        occ_rate_avg  = pw.reducers.avg(pw.this.OccupancyRate),
        capacity      = pw.reducers.max(pw.this.Capacity),
    )
    .with_columns(
        # Price = BASE + ALPHA * avg_occupancy_rate   (bounded to [5, 20])
        price_model1 = pw.apply(
            lambda r: float(np.clip(BASE_PRICE + ALPHA * r, BASE_PRICE*0.5, BASE_PRICE*2)),
            pw.this.occ_rate_avg
        )
    )
)

print("✅ Model 1 Pathway pipeline defined.")

### Model 2 window

Same windowing structure, but now aggregating all five features — occupancy, queue, traffic, special day, vehicle weight. The `@pw.udf` decorator lets me call my Python pricing function inside the Pathway pipeline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  ░░░  MODEL 2 PATHWAY WINDOW  ░░░
#
#  Aggregate demand features per lot per day → demand-based price.
# ─────────────────────────────────────────────────────────────────────────

@pw.udf
def compute_model2_price(occ_rate: float, queue_len: float,
                          traffic_lvl: float, is_special: float,
                          vehicle_weight: float) -> float:
    """Pathway UDF wrapping Model 2 pricing logic."""
    return model2_price(
        occ_rate, queue_len, int(traffic_lvl),
        int(is_special), vehicle_weight
    )


model2_window = (
    stream_with_time
    .windowby(
        pw.this.t,
        instance=pw.this.SystemCodeNumber + "_" + pw.this.day,
        window=pw.temporal.tumbling(datetime.timedelta(days=1)),
        behavior=pw.temporal.exactly_once_behavior()
    )
    .reduce(
        t              = pw.this._pw_window_end,
        lot_id         = pw.reducers.any(pw.this.SystemCodeNumber),
        occ_rate_avg   = pw.reducers.avg(pw.this.OccupancyRate),
        queue_avg      = pw.reducers.avg(pw.this.QueueLength),
        traffic_avg    = pw.reducers.avg(pw.this.TrafficLevel),
        special_any    = pw.reducers.max(pw.this.IsSpecialDay),    # 1 if any special event
        veh_weight_avg = pw.reducers.avg(pw.this.VehicleTypeWeight),
        lat            = pw.reducers.avg(pw.this.Latitude),
        lon            = pw.reducers.avg(pw.this.Longitude),
    )
    .with_columns(
        price_model2 = compute_model2_price(
            pw.this.occ_rate_avg,
            pw.this.queue_avg,
            pw.this.traffic_avg,
            pw.this.special_any,
            pw.this.veh_weight_avg
        )
    )
)

print("✅ Model 2 Pathway pipeline defined.")

---
## Step 5 : Running Pathway

Write both pipeline outputs to CSV, run the stream, then load the results back for plotting.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Collect Model 1 & 2 Pathway outputs into CSV sinks
# ─────────────────────────────────────────────────────────────────────────

os.makedirs('results', exist_ok=True)

pw.io.csv.write(model1_window, 'results/model1_prices.csv')
pw.io.csv.write(model2_window, 'results/model2_prices.csv')

# Run Pathway pipeline
pw.run()

print("✅ Pathway run complete. Results written to results/")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Load Pathway output results
# ─────────────────────────────────────────────────────────────────────────

res1 = pd.read_csv('results/model1_prices.csv')
res2 = pd.read_csv('results/model2_prices.csv')

# Parse timestamps
for rdf in [res1, res2]:
    rdf['t'] = pd.to_datetime(rdf['t'])

print("Model 1 result shape:", res1.shape)
print("Model 2 result shape:", res2.shape)
res2.head()

---
## Step 3 : Model 3 — Competitive pricing

Model 3 needs to compare prices across lots at the same point in time, which is easier to do in pandas than in Pathway. I compute it day by day — for each lot, I look at what nearby lots (within 2 km) are charging that day and adjust accordingly.

Distance is calculated with the Haversine formula using the lat/lon coordinates.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Build per-day lot-level features from the original dataframe
#  (same aggregation logic as Model 2 Pathway window)
# ─────────────────────────────────────────────────────────────────────────

df['Date'] = df['Timestamp'].dt.date

daily_agg = (
    df.groupby(['SystemCodeNumber', 'Date'])
    .agg(
        Capacity         = ('Capacity',          'max'),
        occ_rate_avg     = ('OccupancyRate',      'mean'),
        queue_avg        = ('QueueLength',        'mean'),
        traffic_avg      = ('TrafficLevel',       'mean'),
        special_any      = ('IsSpecialDay',       'max'),
        veh_weight_avg   = ('VehicleTypeWeight',  'mean'),
        Latitude         = ('Latitude',           'first'),
        Longitude        = ('Longitude',          'first'),
    )
    .reset_index()
)

# Compute Model 1 and Model 2 baselines
daily_agg['price_model1'] = daily_agg['occ_rate_avg'].apply(
    lambda r: float(np.clip(BASE_PRICE + ALPHA * r, BASE_PRICE*0.5, BASE_PRICE*2))
)

daily_agg['price_model2'] = daily_agg.apply(
    lambda row: model2_price(
        row['occ_rate_avg'], row['queue_avg'], row['traffic_avg'],
        row['special_any'],  row['veh_weight_avg']
    ), axis=1
)

print("Daily aggregates shape:", daily_agg.shape)
daily_agg.head(6)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  Compute Model 3 prices day by day
# ─────────────────────────────────────────────────────────────────────────

model3_prices  = []
model3_reroutes = []

for date, day_df in daily_agg.groupby('Date'):
    # Build competitor price and location lookups from Model 2 output
    competitor_prices = dict(zip(day_df['SystemCodeNumber'], day_df['price_model2']))
    lot_locations     = dict(zip(day_df['SystemCodeNumber'],
                                  zip(day_df['Latitude'], day_df['Longitude'])))

    for _, row in day_df.iterrows():
        price, reroute = model3_price(
            lot_id         = row['SystemCodeNumber'],
            occ_rate       = row['occ_rate_avg'],
            queue_len      = row['queue_avg'],
            traffic_lvl    = row['traffic_avg'],
            is_special     = row['special_any'],
            vehicle_weight = row['veh_weight_avg'],
            lat            = row['Latitude'],
            lon            = row['Longitude'],
            competitor_prices = competitor_prices,
            lot_locations     = lot_locations,
        )
        model3_prices.append(price)
        model3_reroutes.append(reroute)

daily_agg['price_model3'] = model3_prices
daily_agg['reroute_to']   = model3_reroutes

# Summary of rerouting suggestions
n_reroutes = daily_agg['reroute_to'].notna().sum()
print(f"Model 3 prices computed.")
print(f"Rerouting suggested on {n_reroutes} lot-days ({n_reroutes/len(daily_agg)*100:.1f}%)")
daily_agg[['SystemCodeNumber','Date','price_model1','price_model2','price_model3','reroute_to']].head(10)

---
## Step 7 : Visualizations



### 7.1 Price over time — all lots, all three models

One line per lot. Click on names in the legend to mute/unmute. The dashed gray line is the $10 base for reference.

In [ ]:
output_notebook()

# ─── Colour palette (up to 14 lots) ───
LOTS   = sorted(daily_agg['SystemCodeNumber'].unique())
COLORS = (Category20[20])[:len(LOTS)]
LOT_COLOR = dict(zip(LOTS, COLORS))

# ─── Helper: one line-plot per model ───
def make_price_plot(model_col: str, title: str, y_label: str = 'Price ($)'):
    p = figure(
        title          = title,
        x_axis_type    = 'datetime',
        width          = 900,
        height         = 380,
        tools          = 'pan,wheel_zoom,box_zoom,reset,save',
        toolbar_location = 'above',
    )
    p.yaxis.axis_label = y_label
    p.xaxis.axis_label = 'Date'
    p.add_tools(HoverTool(
        tooltips = [
            ('Lot',   '@lot_id'),
            ('Date',  '@date{%F}'),
            ('Price', '@price{$0.2f}'),
        ],
        formatters = {'@date': 'datetime'},
        mode='mouse'
    ))

    legend_items = []
    for lot, color in LOT_COLOR.items():
        sub = daily_agg[daily_agg['SystemCodeNumber'] == lot].copy()
        sub['Date'] = pd.to_datetime(sub['Date'])
        src = ColumnDataSource(dict(
            date   = sub['Date'],
            price  = sub[model_col],
            lot_id = [lot] * len(sub),
        ))
        ln = p.line('date', 'price', source=src, color=color, line_width=1.8,
                    alpha=0.85, muted_alpha=0.08)
        cr = p.circle('date', 'price', source=src, color=color, size=4, alpha=0.7)
        legend_items.append((lot, [ln]))

    legend = Legend(items=legend_items, location='top_left',
                    click_policy='mute', label_text_font_size='9pt')
    p.add_layout(legend, 'right')
    # Reference base-price line
    p.line([daily_agg['Date'].min(), daily_agg['Date'].max()],
           [BASE_PRICE, BASE_PRICE],
           line_dash='dashed', color='gray', line_width=1, alpha=0.5,
           legend_label='Base $10')
    p.y_range.start = 4
    p.y_range.end   = 26
    return p

daily_agg['Date'] = pd.to_datetime(daily_agg['Date'])

p1 = make_price_plot('price_model1', '📊 Model 1 — Baseline Linear Pricing')
p2 = make_price_plot('price_model2', '📊 Model 2 — Demand-Based Pricing')
p3 = make_price_plot('price_model3', '📊 Model 3 — Competitive Pricing')

grid = column(p1, p2, p3)
show(grid)

### 7.2 All three models on one lot

Useful to see how much the models actually differ from each other on the same lot. Model 1 is the flattest, Model 3 has the most variation.

In [ ]:
# Pick a representative lot for head-to-head comparison
SELECTED_LOT = LOTS[0]

lot_df = daily_agg[daily_agg['SystemCodeNumber'] == SELECTED_LOT].copy()
lot_df['Date'] = pd.to_datetime(lot_df['Date'])

src = ColumnDataSource(dict(
    date    = lot_df['Date'],
    m1      = lot_df['price_model1'],
    m2      = lot_df['price_model2'],
    m3      = lot_df['price_model3'],
    occ     = lot_df['occ_rate_avg'],
))

p = figure(
    title        = f'Model Comparison — {SELECTED_LOT}',
    x_axis_type  = 'datetime',
    width        = 900,
    height       = 420,
    tools        = 'pan,wheel_zoom,reset,save',
    toolbar_location = 'above',
)
p.yaxis.axis_label = 'Price ($)'
p.xaxis.axis_label = 'Date'

p.add_tools(HoverTool(
    tooltips=[
        ('Date',      '@date{%F}'),
        ('Model 1',   '@m1{$0.2f}'),
        ('Model 2',   '@m2{$0.2f}'),
        ('Model 3',   '@m3{$0.2f}'),
        ('Occ Rate',  '@occ{0.2f}'),
    ],
    formatters={'@date': 'datetime'},
    mode='vline'
))

l1 = p.line('date', 'm1', source=src, color='#1f77b4', line_width=2,
             legend_label='Model 1 (Linear)')
l2 = p.line('date', 'm2', source=src, color='#ff7f0e', line_width=2,
             legend_label='Model 2 (Demand)')
l3 = p.line('date', 'm3', source=src, color='#2ca02c', line_width=2,
             legend_label='Model 3 (Competitive)')
p.line([lot_df['Date'].min(), lot_df['Date'].max()],
       [BASE_PRICE, BASE_PRICE],
       line_dash='dotted', color='gray', line_width=1.5,
       legend_label='Base $10')

p.legend.location   = 'top_left'
p.legend.click_policy = 'mute'
p.y_range.start = 4
p.y_range.end   = 26

show(p)

### 7.3 Occupancy vs price scatter (Model 2)

Quick sanity check — higher occupancy should generally push the price up. Each color is a different lot.

In [ ]:
src_scatter = ColumnDataSource(dict(
    occ_rate  = daily_agg['occ_rate_avg'],
    price_m2  = daily_agg['price_model2'],
    price_m3  = daily_agg['price_model3'],
    lot       = daily_agg['SystemCodeNumber'],
    special   = daily_agg['special_any'].astype(str),
))

p_scat = figure(
    title        = 'Occupancy Rate vs Price — All Lots (Model 2)',
    width        = 700,
    height       = 420,
    tools        = 'pan,wheel_zoom,reset,save',
    toolbar_location = 'above',
)
p_scat.xaxis.axis_label = 'Average Daily Occupancy Rate'
p_scat.yaxis.axis_label = 'Price ($)'
p_scat.add_tools(HoverTool(
    tooltips=[('Lot', '@lot'), ('Occ Rate', '@occ_rate{0.2f}'),
              ('Price M2', '@price_m2{$0.2f}'), ('Special', '@special')],
    mode='mouse'
))

# Color by lot
for lot, color in LOT_COLOR.items():
    sub = daily_agg[daily_agg['SystemCodeNumber'] == lot]
    p_scat.circle(
        sub['occ_rate_avg'], sub['price_model2'],
        color=color, size=7, alpha=0.6, legend_label=lot,
        muted_alpha=0.05
    )

p_scat.legend.location    = 'top_left'
p_scat.legend.click_policy = 'mute'
p_scat.legend.label_text_font_size = '9pt'

show(p_scat)

### 7.4 Rerouting events — Model 3

When a lot is ≥95% full and a nearby lot is significantly cheaper, Model 3 flags it for rerouting. This shows how many days each lot hit that condition.

In [ ]:
reroute_df = daily_agg[daily_agg['reroute_to'].notna()].copy()

print(f"Total rerouting events: {len(reroute_df)}")
print("\nReroute frequency by source lot:")
print(reroute_df.groupby('SystemCodeNumber')['reroute_to'].count())

# Bar chart of rerouting events per lot
reroute_counts = reroute_df.groupby('SystemCodeNumber').size().reset_index(name='count')

p_rr = figure(
    x_range  = reroute_counts['SystemCodeNumber'].tolist(),
    title    = 'Model 3 — Rerouting Suggestion Count per Lot',
    width    = 700,
    height   = 350,
    tools    = 'save',
)
p_rr.vbar(
    x    = reroute_counts['SystemCodeNumber'],
    top  = reroute_counts['count'],
    width= 0.7,
    color= '#e74c3c',
    alpha= 0.8,
)
p_rr.xaxis.major_label_orientation = 1.0
p_rr.yaxis.axis_label = 'Number of Rerouting Days'
p_rr.xaxis.axis_label = 'Parking Lot ID'
show(p_rr)

---
## Step 8 : Notes and assumptions

### Demand function (Model 2)

$$
\text{Demand} = 0.6 \cdot \frac{\text{Occupancy}}{\text{Capacity}} + 0.1 \cdot \text{QueueLength} - 0.05 \cdot \text{TrafficLevel} + 0.3 \cdot \text{IsSpecialDay} + 0.1 \cdot \text{VehicleWeight}
$$

$$
\text{Price}_t = \$10 \times \left(1 + 0.5 \times \hat{D}\right) \qquad \hat{D} \in [-0.5, +0.5]
$$

### What I assumed and why

| Assumption | Why |
|------------|-----|
| Base price = \$10 | Given in the problem |
| Traffic reduces demand | High congestion means fewer drivers arriving |
| Truck weight = 1.5, bike = 0.5 | Trucks take more physical space per spot |
| 2 km proximity radius | Reasonable distance a driver would consider switching lots |
| Prices capped at [\$5, \$25] | Keeps things realistic — no crazy spikes |
| Daily tumbling window | Prices update once per day, simple and stable |

### How prices actually move

- Lot nearly empty, no queue, clear roads → price drops toward $5
- Around 60% full with a small queue → stays near $10
- Nearly full on a special day → climbs toward $15–$20
- Model 3 on top: nearby lots more expensive → nudge up slightly; nearby lots cheaper → come down a bit

Changes are gradual — I only take 30% of the gap from competitor prices, so there are no sudden jumps.

In [ ]:
# ─── Final comparison table ───
summary = (
    daily_agg
    .groupby('SystemCodeNumber')[['price_model1','price_model2','price_model3']]
    .agg(['mean','min','max'])
    .round(2)
)
summary.columns = ['_'.join(c) for c in summary.columns]
print("\n📋 Price Summary per Lot across 73 Days")
summary